# Day 1 — Building an N-gram Language Model from Scratch

This notebook turns `docs/day1/theory.md` into working code. We build a model that can
**score** text (probability), **generate** text (Shannon's dice), and report **perplexity**
on held-out text, trained on the Brown Corpus (~1M words).

**How to work through this notebook:**

1. Run cells top to bottom.
2. Five functions are left for **you** to implement — marked `TODO 1` … `TODO 5`.
3. After each TODO there is a **check cell**. The checks use the *cat corpus* — the same
   three sentences from the theory doc — so every expected number is one we already
   computed by hand. If the check cell prints ✓, your code is right. If it throws, read
   the error, fix, re-run.
4. Nothing after a TODO works until you've filled it in. That's on purpose.

| Step | Builds | Theory section |
|---|---|---|
| 1 | Load Brown, train/test split | "held-out is non-negotiable" (7.5) |
| 2 | Vocabulary + `⟨unk⟩` | new mini-lesson below |
| 3 | **TODO 1**: count n-grams | the count table (7.3) |
| 4 | **TODO 2**: MLE probability | "count pairs, divide" (7.3) |
| 5 | watch it crash | "the dog ran" (7.4) |
| 6 | **TODO 3**: add-k smoothing | the fix (7.4) |
| 7 | **TODO 4**: perplexity | the scorecard (7.5) |
| 8 | **TODO 5**: generation | Shannon's dice (4.4) |
| 9 | the finale: our own staircase | Shannon's staircase (4.2) |

In [1]:
%pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 1.5 MB 3.1 MB/s eta 0:00:01
     |████████████████████████████████| 288 kB 9.1 MB/s eta 0:00:01
     |████████████████████████████████| 309 kB 16.4 MB/s eta 0:00:01
     |████████████████████████████████| 98 kB 7.1 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from collections import Counter
import math
import random

import nltk
nltk.download("brown", quiet=True)
from nltk.corpus import brown

random.seed(42)  # same results every run

## Step 1 — Load the corpus and split it

The Brown Corpus (1961): ~57,000 sentences, ~1.16M words, already split into sentences
and words for us. We lowercase everything (so "The" and "the" are one word) and split
**90% train / 10% test**.

We split the data set for scoring the model because testing on its own training data is
letting a student grade their own exam. 

In [5]:
sentences = [[w.lower() for w in sent] for sent in brown.sents()]
random.shuffle(sentences)

split = int(0.9 * len(sentences))
train_sents_raw = sentences[:split]
test_sents_raw  = sentences[split:]

n_train_tokens = sum(len(s) for s in train_sents_raw)
print(f"train: {len(train_sents_raw):,} sentences, {n_train_tokens:,} tokens")
print(f"test:  {len(test_sents_raw):,} sentences")
print("sample:", " ".join(train_sents_raw[0][:15]), "...")

train: 51,606 sentences, 1,047,420 tokens
test:  5,734 sentences
sample: the name fell with lazy affectionate remembrance from her lips . ...


## Step 2 — Vocabulary and the `⟨unk⟩` trick

Here's a problem that we didnt covered in the medium article, because it only appears when you
touch real data. Smoothing handles unseen *pairs* of known words. But we cannot cover all the words like 
the **words the model has never seen at all** — names, typos, rare words. They
aren't in any count table; smoothing can't help because the word isn't even in the
vocabulary.

The standard fix: **decide on a fixed vocabulary up front, and replace everything
outside it with one special token `⟨unk⟩` ("unknown").**

- Words appearing **fewer than 2 times** in training → replaced by `⟨unk⟩` in training data.
- Any test word not in the vocabulary → also mapped to `⟨unk⟩`.

Now "unknown word" is itself a word the model has statistics for — it learned during
training how often *some rare word* appears and in what contexts. Every real LM has a
version of this; modern LLMs solve it more elegantly with subword tokenization,
where any string can be spelled out of pieces.

In [6]:
UNK = "⟨unk⟩"
MIN_COUNT = 2

word_freq = Counter(w for sent in train_sents_raw for w in sent)
vocab = {w for w, c in word_freq.items() if c >= MIN_COUNT}

def apply_vocab(sents):
    return [[w if w in vocab else UNK for w in s] for s in sents]

train_sents = apply_vocab(train_sents_raw)
test_sents  = apply_vocab(test_sents_raw)

pct_unk = 100 * sum(w == UNK for s in train_sents for w in s) / n_train_tokens
print(f"vocabulary: {len(vocab):,} words (from {len(word_freq):,} distinct)")
print(f"{pct_unk:.1f}% of training tokens became {UNK}")

vocabulary: 26,399 words (from 47,581 distinct)
2.0% of training tokens became ⟨unk⟩


## The cat corpus — our unit tests

Before touching Brown, every function you write gets checked against the cat corpus:

```
the cat sat
the cat ran
the dog sat
```

We already computed all its numbers **by hand** in the theory doc — the count table,
P(cat|the) = 2/3 because "the" appear 3 time and 2 times it had cat, simillarly for p(dog|the) = 1/3

Now id we calculate the  P(ran | dog) = count("dog ran") / count("dog") = 0 / 1 = 0. This wrong because if something didnt appear in dataset. Doesnt mean it wont!!
So we do? 
The add-one fix, in one sentence: pretend every possible word pair happened one more time than it really did. So "dog ran" — really seen 0 times — is treated as seen 1 time. "dog sat" — really seen 1 time — is treated as seen 2 times. Every pair gets +1, seen or not.


But now the bottom of the fraction must change too. Here's why. Before smoothing, P(something | dog) divided by 3... no wait — by count("dog") = 1, and the probabilities of everything after "dog" summed neatly to 1 (sat was 1/1, everything else 0). If we inflate all the tops with +1 but keep dividing by 1, the "probabilities" would sum to way more than 1 — nonsense. Since we handed out one fake count to each possible next word, and there are V possible next words, the bottom must grow by V to stay honest.

So what is V here? Count the words that could possibly come next: the, cat, dog, sat, ran — that's 5 — plus one more that's easy to forget: ⟨/s⟩, the end-of-sentence marker, because "the sentence just ends here" is always a possible next event. So V = 6. (Why isn't ⟨s⟩ on the list? Because nothing ever comes before ⟨s⟩ predicts it — it's never a "next word." It only ever sits at the start.)

The add-one values are 1/7 and 2/7. If your code reproduces them, it's correct.

Provided below: the corpus, and the padding helper (`⟨s⟩` start / `⟨/s⟩` end markers —
an n-gram model needs n−1 start markers so the first real word has a full-length context).

Because an n-gram model asks a fixed-size question — "given the last n−1 words, what's next?" — and padding adds exactly n−1 fake words so the question is askable from the very first real word.


In [8]:
CAT = [["the", "cat", "sat"],
       ["the", "cat", "ran"],
       ["the", "dog", "sat"]]

S, E = "⟨s⟩", "⟨/s⟩"

def pad(tokens, n):
    """Wrap a sentence in boundary markers for an n-gram model.
    pad(["the","cat","sat"], 2) -> ["⟨s⟩","the","cat","sat","⟨/s⟩"]
    pad(["the","cat","sat"], 3) -> ["⟨s⟩","⟨s⟩","the","cat","sat","⟨/s⟩"]
    """
    return [S] * (n - 1) + list(tokens) + [E]

print(pad(CAT[0], 2))
print(pad(CAT[0], 3))

['⟨s⟩', 'the', 'cat', 'sat', '⟨/s⟩']
['⟨s⟩', '⟨s⟩', 'the', 'cat', 'sat', '⟨/s⟩']


## Step 3 — TODO 1: count the n-grams

Training an n-gram model = filling two count tables (theory 7.3):

- `ngram_counts`: how many times each **(context, word)** window appears.
  Key = a tuple of n tokens, e.g. `("the", "cat")` for a bigram model.
- `context_counts`: how many times each **context** appears.
  Key = a tuple of n−1 tokens, e.g. `("the",)`. (For a unigram model, n=1, the
  context is the empty tuple `()` — the code below should handle that for free.)

**Recipe:** for each sentence, pad it, then slide a window over it: at every position
`i` from `n-1` to the end, the context is the `n-1` tokens before position `i` and the
word is `tokens[i]`. Count both.

In [ ]:

def count_ngrams(sents, n):
    """Count n-grams in a list of (unpadded) sentences.

    Returns:
        ngram_counts:   Counter mapping tuple of n tokens -> count
        context_counts: Counter mapping tuple of n-1 tokens -> count
    """
    ngram_counts, context_counts = Counter(), Counter()
    for sent in sents:
        tokens = pad(sent, n)
        context = []
        for i in range(len(tokens)):
            context.append(tokens[i])
            if(len(context) == n):
                ngram_counts[tuple(context)] += 1
                context.pop(0)
            if(len(context) == n-1):
                context_counts[tuple(context)] += 1
    return ngram_counts, context_counts




In [49]:
# CHECK 1 — must match the count table in theory.md section 7.3
bi, ctx = count_ngrams(CAT, 2)
assert bi[(S, "the")]   == 3, f"count(⟨s⟩ the) should be 3, got {bi[(S,'the')]}"
assert bi[("the","cat")] == 2
assert bi[("the","dog")] == 1
assert bi[("cat","sat")] == 1
assert bi[("cat","ran")] == 1
assert bi[("dog","sat")] == 1
assert bi[("sat", E)]    == 2
assert bi[("ran", E)]    == 1
assert ctx[("the",)] == 3 and ctx[("cat",)] == 2 and ctx[("dog",)] == 1
assert ctx[(S,)] == 3

# unigram sanity: context is the empty tuple
uni, uctx = count_ngrams(CAT, 1)
assert uctx[()] == 12, "unigram context count = total tokens incl. 3 ⟨/s⟩ = 12"

print("✓ CHECK 1 passed — your counts match the hand-built table")

✓ CHECK 1 passed — your counts match the hand-built table


## Step 4 — TODO 2: MLE probability = divide

Theory 7.3: *"out of all the times 'the' appeared, how often was 'cat' the next word?"*

$$P(\text{word} \mid \text{context}) = \frac{\text{count}(\text{context} + \text{word})}{\text{count}(\text{context})}$$

Edge case: if the context itself was never seen, return 0.0 (don't divide by zero).

In [59]:
def mle_prob(word, context, ngram_counts, context_counts):
    """P(word | context) by maximum likelihood. context is a tuple."""
    return 0.0 if context_counts[context] == 0 else ngram_counts[context + (word,)]/context_counts[context]

In [60]:
# CHECK 2 — the trained model from theory.md 7.3
assert math.isclose(mle_prob("cat", ("the",), bi, ctx), 2/3)
assert math.isclose(mle_prob("dog", ("the",), bi, ctx), 1/3)
assert math.isclose(mle_prob("sat", ("cat",), bi, ctx), 1/2)
assert math.isclose(mle_prob("the", (S,),     bi, ctx), 1.0)
assert mle_prob("ran", ("dog",), bi, ctx) == 0.0   # the famous zero
assert mle_prob("cat", ("purple",), bi, ctx) == 0.0  # unseen context

print("✓ CHECK 2 passed — P(cat|the)=2/3, P(ran|dog)=0")

✓ CHECK 2 passed — P(cat|the)=2/3, P(ran|dog)=0


## Step 5 — Watch it crash (provided)

Theory 7.4, live: score **"the dog ran"** — perfectly good English — with the MLE model.
Then do the same thing at scale: train on Brown and see how many *real* test sentences
the model calls "impossible."

In [61]:
def sentence_prob_mle(sent, n, ngram_counts, context_counts):
    p = 1.0
    tokens = pad(sent, n)
    for i in range(n - 1, len(tokens)):
        context, word = tuple(tokens[i - n + 1:i]), tokens[i]
        p *= mle_prob(word, context, ngram_counts, context_counts)
    return p

print('P("the dog ran") on the cat corpus:',
      sentence_prob_mle(["the", "dog", "ran"], 2, bi, ctx))

# now at scale: bigram MLE on Brown
brown_bi, brown_ctx = count_ngrams(train_sents, 2)
n_zero = sum(sentence_prob_mle(s, 2, brown_bi, brown_ctx) == 0.0
             for s in test_sents[:500])
print(f"{n_zero} of the first 500 real test sentences get probability 0")
print("every one of them would make perplexity infinite")

P("the dog ran") on the cat corpus: 0.0
445 of the first 500 real test sentences get probability 0
every one of them would make perplexity infinite


## Step 6 — TODO 3: add-k smoothing

Theory 7.4's fix: pretend every possible pair occurred k more times than it did
(k=1 is "add-one"). V is the number of possible next words:

$$P(\text{word} \mid \text{context}) = \frac{\text{count}(\text{context}+\text{word}) + k}{\text{count}(\text{context}) + k \cdot V}$$

Note this works even for a context that was never seen: count 0 + k over 0 + kV = 1/V,
a uniform guess — which is the honest answer when you know nothing.

In [71]:
def add_k_prob(word, context, ngram_counts, context_counts, V, k=1.0):
    """Add-k smoothed P(word | context)."""
    return (ngram_counts[context + (word,)] + k)/(context_counts[context] + k*V)

In [72]:
# CHECK 3 — the numbers from theory.md 7.4 (cat-corpus vocabulary: V = 6)
V_CAT = 6  # the, cat, dog, sat, ran, ⟨/s⟩

assert math.isclose(add_k_prob("ran", ("dog",), bi, ctx, V_CAT, k=1), 1/7), \
    "P(ran|dog) with add-one should be (0+1)/(1+6) = 1/7"
assert math.isclose(add_k_prob("sat", ("dog",), bi, ctx, V_CAT, k=1), 2/7), \
    "P(sat|dog) should fall from 1 to (1+1)/(1+6) = 2/7 — the smoothing tax"
assert math.isclose(add_k_prob("the", (S,), bi, ctx, V_CAT, k=1), 4/9)

print("✓ CHECK 3 passed — 'the dog ran' is now unlikely, not impossible")

✓ CHECK 3 passed — 'the dog ran' is now unlikely, not impossible


## Step 7 — TODO 4: perplexity

Theory 7.5, the scorecard. Recipe:

1. For every test sentence: pad it, and for each position take
   $\log_2 P(\text{word} \mid \text{context})$ using `add_k_prob`.
2. Sum all the logs; count the total number of predictions N (across all sentences).
3. Average surprise $H = -\text{total} / N$, and **perplexity = $2^H$**.
4. If any probability is 0 (can happen when k=0), return `float("inf")`.

**The check value, computed by hand.** Train on the cat corpus, test on
"the dog ran" with add-one. The four factors are numbers we know:
P(the|⟨s⟩)=4/9, P(dog|the)=2/9, P(ran|dog)=1/7, P(⟨/s⟩|ran)=2/7.

$$H = -\tfrac{1}{4}\left(\log_2\tfrac{4}{9} + \log_2\tfrac{2}{9} + \log_2\tfrac{1}{7} + \log_2\tfrac{2}{7}\right) \approx 1.9886 \;\Rightarrow\; \text{PPL} = 2^H \approx 3.97$$

Reading "the dog ran", the smoothed model feels like it's choosing among ~4 options
per word. Your function must reproduce that.

In [80]:
def perplexity(test_sents, n, ngram_counts, context_counts, V, k=1.0):
    """Perplexity of the model on a list of (unpadded) test sentences."""
    N = 0
    entropy = 0
    for test_sent in test_sents:
        test_sent = pad(test_sent, n)
        size = len(test_sent)
        for i in range(n-1, size):
            p = add_k_prob(test_sent[i], tuple(test_sent[i-n+1:i]), ngram_counts, context_counts, V, k)
            entropy += math.log2(p)
            N += 1
    
    H = -(entropy/N)
    perp = 2**H
    return perp

        




In [81]:
# CHECK 4 — the hand-computed value
ppl = perplexity([["the", "dog", "ran"]], 2, bi, ctx, V_CAT, k=1)
assert 3.95 < ppl < 3.99, f"expected ≈ 3.97, got {ppl}"
print(f"✓ CHECK 4 passed — PPL = {ppl:.4f} (hand-computed: 3.9686)")

✓ CHECK 4 passed — PPL = 3.9686 (hand-computed: 3.9686)


## Step 8 — TODO 5: generation (Shannon's dice)

Theory 4.4: a model that can score can also **write**. Recipe:

1. Start with the context `(⟨s⟩,) * (n-1)`.
2. Look at every n-gram in `ngram_counts` that starts with this context; the counts
   are your dice weights. (Sampling from raw counts = sampling from the MLE model.)
3. Roll: `random.choices(words, weights=counts)`.
4. Emit the word, slide the context window forward, repeat until you roll `⟨/s⟩`
   or hit `max_len`.

Hint: build a lookup `continuations[context] -> (list_of_words, list_of_counts)` once
at the top from `ngram_counts`, so each step is fast.

In [ ]:
def generate(n, ngram_counts, max_len=20):
    """Sample one sentence (list of words) from the model."""
    ngram_dict = {}
    for key, val in ngram_counts:
        history_string = " ".join(key[i] for i in range(n - 1))
        if history_string in ngram_dict:
            psuggestion = ngram_dict[history_string]
            psuggestion.append(key[n-1])
        else:
            ngram_dict[history_string] = [key[n-1]]
    
    context = [S] * (n - 1)
    ans = " "
    for _ in range(max_len):
        if " ".join(context) in ngram_dict:
            psuggestions = ngram_dict[history_string]
            psuggestion = random.choices(psuggestions)
            ans.join(psuggestion)
            context.pop(0)
            context.append(psuggestion)
            if(psuggestion == E):
                break
    
    return ans


In [102]:
# CHECK 5 — every generated cat-corpus sentence must be one the dice allow.
# Quick intuition test before you look at the set: can the model generate
# "the dog ran"? No — after "dog" the only continuation it ever saw is "sat",
# so that die has one face. The dice can only produce:
allowed = {"the cat sat", "the cat ran", "the dog sat"}
samples = {" ".join(generate(2, bi)) for _ in range(200)}
assert samples <= allowed, f"impossible sentence generated: {samples - allowed}"
assert len(samples) > 1, "with 200 rolls you should see more than one sentence"
print("✓ CHECK 5 passed — 200 rolls, sentences seen:", samples)

AssertionError: impossible sentence generated: {' '}

## The finale — your own Shannon staircase

Everything works. Now the payoff, in two parts:

**Part 1 — perplexity staircase.** Unigram vs bigram vs trigram on real held-out text,
each scored with add-one (k=1) and with gentle smoothing (k=0.01).

**Part 2 — read the table carefully, it contains two surprises:**

1. **With add-one, the bigram loses to the unigram.** More context made the model
   *worse*?! No — the smoothing tax did. With V ≈ 26,000, add-one hands every context
   26,000 phantom counts, drowning the real bigram counts (most bigram contexts were
   seen only a handful of times). Shrink the tax to k=0.01 and the bigram beats the
   unigram comfortably. Your first-ever hyperparameter tuning.
2. **The trigram loses even at k=0.01.** That's not a smoothing problem — it's a data
   problem: 1M words is simply not enough to see most 3-word combinations even once.
   More context only helps if you have the data to fill it. (Sound familiar? It's the
   scaling story of the entire field, in miniature.)

This one table is why smarter smoothing (Kneser-Ney) had to be invented — and,
eventually, why counting had to be replaced by learning.

In [ ]:
V_BROWN = len(vocab | {UNK, E})
print(f"V = {V_BROWN:,}\n")

models = {}
print(f"{'model':<10} {'k=1':>12} {'k=0.01':>12}")
for n in (1, 2, 3):
    counts_n, ctx_n = count_ngrams(train_sents, n)
    models[n] = (counts_n, ctx_n)
    row = [perplexity(test_sents, n, counts_n, ctx_n, V_BROWN, k=k)
           for k in (1.0, 0.01)]
    name = {1: "unigram", 2: "bigram", 3: "trigram"}[n]
    print(f"{name:<10} {row[0]:>12,.1f} {row[1]:>12,.1f}")

In [ ]:
# Part 2 — Shannon's ladder, generated by YOUR models
for n in (1, 2, 3):
    counts_n, _ = models[n]
    name = {1: "unigram", 2: "bigram", 3: "trigram"}[n]
    print(f"--- {name} ---")
    for _ in range(3):
        print("  ", " ".join(generate(n, counts_n, max_len=15)))
    print()

## What you just proved

- **The zero-count problem is not theoretical.** Roughly 9 in 10 real, grammatical test
  sentences got probability 0 from the pure counting model. Smoothing isn't polish;
  without it the model is unusable.
- **Smoothing is a trade, not a free fix.** Add-one taxed the bigram into losing to the
  unigram; k=0.01 fixed it. Tuning k was your first-ever hyperparameter search.
- **More context needs more data.** The trigram lost on 1M words — the count-table
  approach hits a wall that no k can fix. This wall is what embeddings and neural
  models (Day 4 onward) exist to break through.
- **Generation quality tracks perplexity.** The bigram/trigram samples read locally
  fine and globally lost — exactly Shannon's ladder, exactly the theory doc's promise.

**For the Medium article:** the perplexity table and the generated-samples ladder are
your two money-shots. Screenshot both.

**Next (Day 2):** tokenization — why modern models split words into subword pieces,
which also dissolves the `⟨unk⟩` hack you met today.